In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

ROOT           = Path.cwd().parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
EXCEL_PATH     = ROOT / 'excel' / 'campaign_summary.xlsx'

# Load all processed data
df         = pd.read_csv(DATA_PROCESSED / 'ads_cleaned.csv')
ab_results = pd.read_csv(DATA_PROCESSED / 'ab_test_results.csv')

# Rebuild platform_budget summary
platform_budget = df.groupby('channel_used').agg(
    campaigns     = ('campaign_id', 'count'),
    avg_ctr       = ('ctr', 'mean'),
    avg_roi       = ('roi', 'mean'),
    total_spend   = ('acquisition_cost', 'sum'),
    total_revenue = ('estimated_revenue', 'sum'),
    avg_conv_rate = ('conversion_rate', 'mean'),
).round(4).reset_index()

platform_budget['roas'] = (
    platform_budget['total_revenue'] / platform_budget['total_spend']
).round(3)

print("Data loaded. Ready to build Excel workbook.")

Data loaded. Ready to build Excel workbook.


In [5]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()

# ── Sheet 1: Executive Summary ────────────────────────────────
ws1 = wb.active
ws1.title = 'Executive Summary'

# Title
ws1['A1'] = 'Social Media Campaign Analytics — Executive Summary'
ws1['A1'].font      = Font(bold=True, size=16, color='1F4E79')
ws1['A1'].alignment = Alignment(horizontal='left')
ws1.merge_cells('A1:F1')

# KPI headers
headers = ['Platform', 'Campaigns', 'Avg CTR %', 'Avg CPC $', 'Avg ROI', 'ROAS']
for col, header in enumerate(headers, 1):
    cell            = ws1.cell(row=3, column=col, value=header)
    cell.font       = Font(bold=True, color='FFFFFF')
    cell.fill       = PatternFill('solid', fgColor='1F4E79')
    cell.alignment  = Alignment(horizontal='center')

# Data rows
for row_idx, row in platform_budget.iterrows():
    ws1.cell(row=4+row_idx, column=1, value=row['channel_used'])
    ws1.cell(row=4+row_idx, column=2, value=int(row['campaigns']))
    ws1.cell(row=4+row_idx, column=3, value=round(row['avg_ctr'] * 100, 3))
    ws1.cell(row=4+row_idx, column=4, value=round(row['total_spend'] / row['campaigns'], 2))
    ws1.cell(row=4+row_idx, column=5, value=round(row['avg_roi'], 3))
    ws1.cell(row=4+row_idx, column=6, value=round(row['roas'], 3))
    # Alternate row shading
    if row_idx % 2 == 0:
        for col in range(1, 7):
            ws1.cell(row=4+row_idx, column=col).fill = PatternFill('solid', fgColor='EEF2FF')

# Safe auto-fit columns (handles merged cells)
for col_idx in range(1, ws1.max_column + 1):
    max_len   = 0
    col_letter = get_column_letter(col_idx)
    for row in ws1.iter_rows(min_col=col_idx, max_col=col_idx):
        for cell in row:
            try:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            except:
                pass
    ws1.column_dimensions[col_letter].width = max_len + 4

print("Sheet 1 done: Executive Summary")

# ── Sheet 2: A/B Test Results ─────────────────────────────────
ws2 = wb.create_sheet('AB Test Results')

ws2['A1'] = 'A/B Test Results — Platform Comparison'
ws2['A1'].font = Font(bold=True, size=14, color='3B1F79')
ws2.merge_cells('A1:G1')
ws2.append([])  # blank row

# Write dataframe with styled header
for r_idx, r in enumerate(dataframe_to_rows(ab_results, index=False, header=True)):
    ws2.append(r)
    if r_idx == 0:
        for cell in ws2[ws2.max_row]:
            cell.font      = Font(bold=True, color='FFFFFF')
            cell.fill      = PatternFill('solid', fgColor='3B1F79')
            cell.alignment = Alignment(horizontal='center')

# Colour rows: green = significant, red = not significant
for row in ws2.iter_rows(min_row=4, max_row=ws2.max_row):
    for cell in row:
        if cell.column == 6:  # 'significant' column
            if cell.value == 'Yes':
                for c in row:
                    c.fill = PatternFill('solid', fgColor='D1F2E3')
            elif cell.value == 'No':
                for c in row:
                    c.fill = PatternFill('solid', fgColor='FFE8E8')

# Safe auto-fit
for col_idx in range(1, ws2.max_column + 1):
    max_len    = 0
    col_letter = get_column_letter(col_idx)
    for row in ws2.iter_rows(min_col=col_idx, max_col=col_idx):
        for cell in row:
            try:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            except:
                pass
    ws2.column_dimensions[col_letter].width = max_len + 4

print("Sheet 2 done: AB Test Results")

# ── Sheet 3: Campaign Data ────────────────────────────────────
ws3 = wb.create_sheet('Campaign Data')

cols_to_export = [
    'channel_used', 'campaign_goal', 'customer_segment',
    'ctr', 'cpc', 'roi', 'conversion_rate',
    'impressions', 'clicks'
]

for r_idx, r in enumerate(dataframe_to_rows(df[cols_to_export], index=False, header=True)):
    ws3.append(r)
    if r_idx == 0:
        for cell in ws3[1]:
            cell.font      = Font(bold=True, color='FFFFFF')
            cell.fill      = PatternFill('solid', fgColor='595959')
            cell.alignment = Alignment(horizontal='center')

# Safe auto-fit
for col_idx in range(1, ws3.max_column + 1):
    max_len    = 0
    col_letter = get_column_letter(col_idx)
    for row in ws3.iter_rows(min_col=col_idx, max_col=col_idx):
        for cell in row:
            try:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            except:
                pass
    ws3.column_dimensions[col_letter].width = max_len + 4

print("Sheet 3 done: Campaign Data")

# ── Save ──────────────────────────────────────────────────────
wb.save('../excel/campaign_summary.xlsx')
print("Excel workbook saved to excel/campaign_summary.xlsx")
print(f"Sheets created: {wb.sheetnames}")

Sheet 1 done: Executive Summary
Sheet 2 done: AB Test Results
Sheet 3 done: Campaign Data
Excel workbook saved to excel/campaign_summary.xlsx
Sheets created: ['Executive Summary', 'AB Test Results', 'Campaign Data']
